# Virchow Training on Google Colab (Resume-Safe)

This notebook runs `scripts/train_virchow_preprocessed_colab.py` and stores checkpoints in Google Drive so training can resume after Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!git clone https://github.com/<your-username>/<your-repo>.git GP_ECG
%cd /content/GP_ECG

If you already uploaded your repo into Drive instead of cloning, skip previous cell and set `REPO_DIR` below to that path.

In [ ]:
REPO_DIR = '/content/GP_ECG'
PREPROCESSED_DIR = '/content/drive/MyDrive/GP_ECG_DATA/preprocessed'
RUN_DIR = '/content/drive/MyDrive/GP_ECG_RUNS/virchow_preprocessed_run_01'
HF_TOKEN = ''  # optional: paste token if Virchow access is gated

import os
os.makedirs(RUN_DIR, exist_ok=True)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN

In [ ]:
%cd {REPO_DIR}
!python -m pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install timm h5py tqdm huggingface_hub

Optional if model access is gated: run login once and paste your Hugging Face token.

In [ ]:
# !huggingface-cli login

In [ ]:
%cd {REPO_DIR}
!python scripts/train_virchow_preprocessed_colab.py \
  --preprocessed-dir "{PREPROCESSED_DIR}" \
  --out-dir "{RUN_DIR}" \
  --epochs 10 \
  --batch-size 64 \
  --num-workers 2 \
  --resume \
  --save-every-epoch-copy

In [ ]:
import json, os
print('Run dir:', RUN_DIR)
for name in ['checkpoint_last.pt', 'model_best.pt', 'metrics_history.json', 'metrics_final.json', 'run_progress.json']:
    p = os.path.join(RUN_DIR, name)
    print(('OK ' if os.path.exists(p) else 'MISSING '), p)

hist = os.path.join(RUN_DIR, 'metrics_history.json')
if os.path.exists(hist):
    with open(hist, 'r', encoding='utf-8') as f:
        rows = json.load(f)
    if rows:
        print('Last epoch record:', rows[-1])